① Tool Definitions + Messages
개발자가 사용할 수 있는 함수(예: get_weather(location))를 미리 정의
사용자가 질문: “What’s the weather in Paris?”

② Tool Calls
모델이 질문을 보고 “아, 이건 get_weather("paris") 함수를 호출해야겠네”
텍스트가 아니라 함수 호출 요청(JSON) 을 생성

③ Execute Function Code
실제 코드에서 get_weather("paris") 실행
외부 API(OpenWeather 같은) 호출

④ Results (All Prior Messages)
함수 실행 결과가 다시 모델에게 전달
모델은 이제 “파리의 온도 = 14도”라는 사실을 알게 됨

⑤ Final Response
모델이 사용자에게 자연어로 최종 답변 생성
“It’s currently 14°C in Paris.”

- LLM이 API를 직접 실행하는 게 아니라
“어떤 함수를 호출할지 결정”만 하고
실행은 개발자 코드,
결과를 다시 받아 문장 생성
👉 LLM + 외부 시스템 연동 구조

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
client = OpenAI()
OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')

In [4]:
import requests

city_name ='Seoul'
units = 'metric'

url = f'https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}'
response = requests.get(url)
data = response.json()

weather_info = {}

if response.status_code == 200:
    weather_description = data['weather'][0]['description']
    temp = data['main']['temp']
    temp_feels_like = data['main']['feels_like']
    humidity = data['main']['humidity']

    weather_info = {
        'city': city_name,
        'description': weather_description,
        'temperature': temp,
        'temperature_feels_like': temp_feels_like,
        'humidity': humidity
    }
else:
    weather_info = {
        'city': city_name,
        'description': 'Not Found',
        'temperature': 'Not Found',
        'temperature_feels_like': 'Not Found',
        'humidity': 'Not Found'
    }

weather_info

{'city': 'Seoul',
 'description': 'overcast clouds',
 'temperature': 27.76,
 'temperature_feels_like': 32.84,
 'humidity': 89}

In [20]:
import json

def get_current_weather(city_name='Seoul', units='metric'):
    """
    Args:
        - City: 날씨 정보 가져올 도시 이름   
            - 서울 -> Seoul 
    Return:
        - str: json 형식으로 변환된 현재 날씨 정보
    """
    url = f'https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}'
    response = requests.get(url)
    data = response.json()

    weather_info = {}

    if response.status_code == 200:
        weather_description = data['weather'][0]['description']
        temp = data['main']['temp']
        temp_feels_like = data['main']['feels_like']
        humidity = data['main']['humidity']

        weather_info = {
            'city': city_name,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like': temp_feels_like,
            'humidity': humidity
        }
    else:
        weather_info = {
            'city': city_name,
            'description': 'Not Found',
            'temperature': 'Not Found',
            'temperature_feels_like': 'Not Found',
            'humidity': 'Not Found'
        }

    return json.dumps(weather_info, ensure_ascii=False)

In [21]:
tools_to_execute = {
    'get_current_weather': get_current_weather
}

In [22]:
print(tools_to_execute['get_current_weather'].__doc__)


    Args:
        - City: 날씨 정보 가져올 도시 이름   
            - 서울 -> Seoul 
    Return:
        - str: json 형식으로 변환된 현재 날씨 정보
    


In [23]:
from pprint import pprint

def run_converation(user_prompt, model='gpt-5.6-luna'):
    messages = [
        {'role': 'system', 'content': '사용자 요구 분석해 직접 대답하거나 주어진 함수 이용해 필요 정보 확보한 후 대답.'},
        {'role': 'user', 'content': user_prompt}
    ]

    tools = [
        {
            'type': 'function',
            'function': {
                'name': 'get_current_weather',
                'description': tools_to_execute['get_current_weather'].__doc__,
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'city_name': {
                            'type': 'string',
                            'description': '''
                                - 날씨 정보 가져올 도시 이름. (필수값, 영문)
                                    - 서울 -> Seoul
                                    - 부산 -> Busan
                            '''
                        },
                        'units': {
                            'type': 'string',
                            'description': '''
                                - 온도 단위 설정 문자열
                                    - metric (기본값: 섭씨, 미터)
                                    - imperial (화씨, 야드)
                            ''',
                            'enum': ['metric', 'imperial']
                        }
                    },
                    'required': ['city_name']
                }
            }
        }
    ]

    response1 = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        reasoning_effort='none'
    )

    response1_message = response1.choices[0].message
    response1_tool_calls = response1_message.tool_calls

    if response1_tool_calls:
        messages.append(response1_message)

        for tool_call in response1_tool_calls:
            function_name = tool_call.function.name
            print(f'[tool] {function_name} 호출')

            function_to_execute = tools_to_execute[function_name]
            function_args = json.loads(tool_call.function.arguments)
            function_response = function_to_execute(**function_args)

            messages.append({
                'role': 'tool',
                'tool_call_id': tool_call.id,
                'name': function_name,
                'content': function_response
            })
            pprint(messages)

        response2 = client.chat.completions.create(
            model=model,
            messages=messages
        )

        return response2.choices[0].message.content
    
    else:
        return response1_message.content


In [24]:
run_converation('오늘 서울 날씨 확인')

[tool] get_current_weather 호출
[{'content': '사용자 요구 분석해 직접 대답하거나 주어진 함수 이용해 필요 정보 확보한 후 대답.',
  'role': 'system'},
 {'content': '오늘 서울 날씨 확인', 'role': 'user'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_3t2v5xcGDyzNwk3COvLyYt8H', function=Function(arguments='{"city_name":"Seoul","units":"metric"}', name='get_current_weather'), type='function')]),
 {'content': '{"city": "Seoul", "description": "overcast clouds", '
             '"temperature": 27.76, "temperature_feels_like": 33.64, '
             '"humidity": 94}',
  'name': 'get_current_weather',
  'role': 'tool',
  'tool_call_id': 'call_3t2v5xcGDyzNwk3COvLyYt8H'}]


'오늘 서울은 **흐리고 현재 약 27.8°C**입니다.  \n체감온도는 **약 33.6°C**, 습도는 **94%**로 매우 후텁지근하겠습니다.'

In [25]:
print(run_converation('금일 경기도 하남시 날씨 확인'))

[tool] get_current_weather 호출
[{'content': '사용자 요구 분석해 직접 대답하거나 주어진 함수 이용해 필요 정보 확보한 후 대답.',
  'role': 'system'},
 {'content': '금일 경기도 하남시 날씨 확인', 'role': 'user'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_XkpyTYzQgTgAQNnewHGHIBuj', function=Function(arguments='{"city_name":"Hanam","units":"metric"}', name='get_current_weather'), type='function')]),
 {'content': '{"city": "Hanam", "description": "overcast clouds", '
             '"temperature": 27.9, "temperature_feels_like": 32.3, "humidity": '
             '83}',
  'name': 'get_current_weather',
  'role': 'tool',
  'tool_call_id': 'call_XkpyTYzQgTgAQNnewHGHIBuj'}]
금일 경기도 하남시는 **흐린 날씨**이며, 현재 기온은 약 **27.9°C**, 체감온도는 **32.3°C**, 습도는 **83%**입니다.  
덥고 습하니 외출 시 수분 섭취와 더위에 유의하세요.
